# ML · Encuentro 3 · Agrupación, PCA y ensambles

Encontrar estructura y combinar modelos

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Parecidos no son categorías de calidad](#scrollTo=tema-ml-clustering)
2. [Resumir sin inventar clases](#scrollTo=tema-ml-pca)
3. [Varias respuestas, una comparación justa](#scrollTo=tema-ml-ensambles)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · DBSCAN: agrupar por densidad

**Lo que vas a poder hacer:** Construir una segmentación exploratoria para organizar una revisión.

Comparar centros de acopio por sus indicadores ayuda a organizar una revisión cuando no existen etiquetas de grupo.

### Entiende la idea

Agrupar significa reunir casos que se parecen en las medidas que elegimos. Aquí cada caso es un centro de acopio y miramos dos cosas: cuánto espacio usa y cuántos lotes tiene pendientes. No recibimos una respuesta que diga a qué grupo pertenece cada centro; la buscamos por semejanza. A esta tarea se la llama clustering o agrupación. DBSCAN (Density-Based Spatial Clustering of Applications with Noise; agrupación espacial basada en densidad con casos no agrupados) es una forma de hacerlo: conecta centros cercanos cuando hay suficientes vecinos. Nosotros elegimos qué tan cerca deben estar y cuántos vecinos bastan. El grupo organiza una revisión; no es una etiqueta de calidad.

### Un caso pequeño antes del programa

**Cercanía antes de poner nombres**

Seis centros de acopio ficticios entregan dos porcentajes del mismo periodo. X es capacidad de almacenamiento utilizada: volumen ocupado / capacidad total × 100. Y es proporción de lotes pendientes de revisión: lotes pendientes / lotes recibidos × 100. Comparamos ambos indicadores con el mismo peso, una elección didáctica que habría que justificar en una operación. El radio 3 se mide en este plano de puntos porcentuales. Ninguna fila representa a una persona.

| Centro | X · Capacidad utilizada (%) | Y · Lotes pendientes (%) |
| --- | --- | --- |
| A | 20 | 20 |
| B | 22 | 20 |
| C | 20 | 22 |
| D | 70 | 60 |
| E | 72 | 60 |
| F | 5 | 90 |

1. Con radio 3, A = (20,20) y B = (22,20) están a √[(22−20)² + (20−20)²] = 2. Son vecinos. A y C también distan 2. El mínimo elegido es 2, contando al propio punto.
2. A, B y C forman un grupo conectado. D = (70,60) y E = (72,60) distan 2 y forman otro. F = (5,90) no alcanza el mínimo de vecinos dentro del radio.
3. La lista de revisión queda: grupo 1, A–B–C; grupo 2, D–E; F, revisión individual. El número del grupo es un identificador: no significa mejor o peor calidad.

**Qué demuestra:** Podemos organizar la exploración en dos perfiles y una revisión individual. Antes de decidir una acción para F se consultarían sus registros: estar sin grupo no demuestra una falla ni autoriza borrar datos.

### Ahora identifica los datos de esta práctica

Relaciona cada color con su grupo y cuenta los puntos «No agrupado». Son 123 puntos simulados; el explorador del aula utiliza otros 29 centros.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Cambia solo eps, el radio de cercanía. Conserva los puntos y el mínimo de vecinos para poder atribuir el cambio a una sola decisión.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


In [ ]:
# Generador local: repetir este bloque conserva los mismos puntos al cambiar eps.
rng_grupos = np.random.default_rng(42)
a=rng_grupos.normal([0,0],.25,(60,2));b=rng_grupos.normal([2,2],.25,(60,2));aislados=np.array([[-2,2],[4,-1],[0,4]])
datos=np.vstack([a,b,aislados]);escalados=StandardScaler().fit_transform(datos)
etiquetas=DBSCAN(eps=.35,min_samples=5).fit_predict(escalados)
display(pd.Series(etiquetas).value_counts().sort_index().rename('número de puntos'))
for grupo in np.unique(etiquetas):
    sel=etiquetas==grupo;plt.scatter(datos[sel,0],datos[sel,1],label='No agrupado' if grupo==-1 else f'Grupo {grupo}')
plt.xlabel('Medida 1 (simulada)');plt.ylabel('Medida 2 (simulada)');plt.title('DBSCAN: agrupación por densidad');plt.legend();plt.show()


### Lectura del gráfico 4 · Del color a una decisión de revisión

Cada punto es una observación con dos números inventados: medida 1 en horizontal y medida 2 en vertical. El color se lee con la **leyenda**: Grupo 0 y Grupo 1 son nombres, mientras «No agrupado» corresponde a −1. Ninguno significa bueno o malo.

La tabla de conteos tiene una fila por etiqueta: el número de la etiqueta y cuántos puntos recibió. La suma debe ser **123**. Comprueba los conteos que imprime tu ejecución. La semilla local conserva estos puntos al repetir el bloque; cambiarla o modificar cómo se generan sí crearía otra simulación.

**Una separación importante:** se estandarizaron las medidas antes de agrupar, para compararlas según su propia dispersión. El dibujo muestra las coordenadas originales. Por eso el radio 0,35 no se puede medir directamente con una regla sobre estos ejes. Cambiar la escala también cambia lo que significa parecerse.

**Prueba pequeña manteniendo los puntos:** guarda los conteos actuales. En tu copia cambia únicamente `eps=.35` a `eps=.50` y vuelve a ejecutar este bloque. El generador local reproduce los mismos `datos`: solo cambió la cercanía permitida. Compara las nuevas etiquetas y conteos con los anteriores. El número de grupos puede crecer, reducirse o fusionarse al cambiar configuraciones; lo que debes documentar es qué casos se revisan juntos y cuáles aparte.

Antes de dar una acción a un grupo, vuelve a las medidas originales. En este conjunto no hay significado operativo real que permita diagnosticar una falla: sirve para practicar agrupación, no para decidir manejo agrícola.


### Lee, comprueba y explica

Relaciona cada color con su grupo y cuenta los puntos «No agrupado». Son 123 puntos simulados; el explorador del aula utiliza otros 29 centros.

**Prueba pequeña:** Cambia solo eps, el radio de cercanía. Conserva los puntos y el mínimo de vecinos para poder atribuir el cambio a una sola decisión.

**Vuelve al caso:** Un grupo significa cercanía con las variables y parámetros elegidos. Antes de darle un significado operativo, revisa qué comparten sus registros.

**Pregunta:** ¿Por qué un grupo de DBSCAN no equivale a una clase real ni a una recomendación agrícola?

**Respuesta razonada:** El grupo surge de las variables, su escala y los parámetros elegidos. Hay que investigar qué comparten sus miembros y comprobar si esa agrupación ayuda en la operación. Un punto sin grupo tampoco es automáticamente un dato erróneo.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-clustering)


## Tema 2 · Representar los mismos casos con componentes

**Lo que vas a poder hacer:** Construir una vista resumida de registros y documentar la información que conserva.

Resumir medidas que cambian juntas facilita una gráfica; debemos saber qué información se pierde.

### Entiende la idea

Imagina que anotas dos indicadores de cada centro y ambos suben casi al mismo tiempo. Quizá puedas resumir parte de esa información sin cargar con todos los números. PCA (Principal Component Analysis; análisis de componentes principales) es un método que busca ese resumen combinando medidas. Una componente es una nueva medida calculada con las originales; no es un centro nuevo. Para entenderla dibujaremos cuatro centros y buscaremos una dirección que los represente. Después comprobaremos cuánto de sus diferencias se conserva y cuánto se pierde. Ese porcentaje no mide aciertos de una predicción.

### Un caso pequeño antes del programa

**Dos medidas que se mueven juntas**

Otros cuatro centros ficticios, G–J, tienen estos porcentajes inventados de capacidad utilizada y lotes pendientes. En ambas columnas la media es 50 %. Restarla produce las columnas centradas: 48−50 = −2 puntos porcentuales. Los valores negativos describen diferencias respecto a la media, no porcentajes de ocupación negativos. Elegimos columnas que cambian juntas para ver la cuenta completa.

| Centro | Ocupación (%) | Pendientes (%) | X centrado (puntos porcentuales) | Y centrado (puntos porcentuales) |
| --- | --- | --- | --- | --- |
| G | 48 | 48 | −2 | −2 |
| H | 49 | 49 | −1 | −1 |
| I | 51 | 51 | 1 | 1 |
| J | 52 | 52 | 2 | 2 |

1. Calcula el centro: (48+49+51+52)/4 = 50 en cada columna. Las diferencias son −2, −1, 1 y 2. Su suma de cuadrados es 4+1+1+4 = 10 por columna; entre ambas, 20. Para comparar proporciones no necesitamos dividir por el mismo denominador de varianza.
2. Si conservamos solo la coordenada X, retenemos 10 de los 20 puntos porcentuales al cuadrado de la suma total: 10/20 = 50 %. Se descarta la coordenada Y; en esta tabla se puede reconstruir porque Y = X, pero esa coincidencia no ocurre en general.
3. La dirección diagonal de 45° contiene todos estos puntos. Proyectarlos sobre ella no cambia su posición: conserva el 100 % de la variación en este ejemplo especialmente construido.

**Qué demuestra:** Una sola dirección resume estas dos columnas sin perder variación en los cuatro registros elegidos. Eso no demuestra que siempre baste una componente ni que podamos predecir calidad con 100 % de exactitud.

### Ahora identifica los datos de esta práctica

Sigue un punto desde sus tres medidas hasta PC1. Lee los porcentajes que calcula tu ejecución; no son los del ejemplo pequeño del aula.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Una coordenada negativa indica posición respecto al centro. No significa rendimiento negativo ni mala calidad.


### PCA y sus coordenadas, sin confundir porcentajes

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **PCA** — PCA: Principal Component Analysis (Análisis de componentes principales). Método que combina medidas numéricas en nuevas direcciones perpendiculares entre sí, ordenadas para conservar tanta variación como sea posible. Trabaja con datos centrados; decidir si se estandarizan antes es otra operación. El porcentaje conservado describe variación de las entradas, no aciertos de clasificación ni utilidad garantizada para predecir. [Fuente](https://scikit-learn.org/stable/modules/decomposition.html#pca)

- **PC1 y PC2** — PC1 y PC2: Principal Component 1 / Principal Component 2 (Primera y segunda componente principal). PC1 es la primera nueva dirección de PCA y PC2 la siguiente, perpendicular a la primera. La coordenada de un caso se obtiene al combinar sus medidas preparadas con los coeficientes de esa dirección. Son ejes calculados, no columnas físicas originales ni categorías. Los coeficientes pueden cambiar de signo junto con las coordenadas sin cambiar la representación geométrica; al hablar de «cargas» hay que declarar la convención. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

- **Varianza** — Varianza: Dispersión cuadrática respecto a la media. Resume cuánto se separan los valores de su media: se elevan al cuadrado las diferencias y se suman. La varianza poblacional divide por n; la estimación muestral habitual divide por n−1, con n mayor que uno. Tiene las unidades originales al cuadrado y vale cero si todos los valores son iguales. Debe declararse el denominador: StandardScaler usa n y explained_variance_ de PCA usa n−1; las proporciones coinciden si el mismo denominador se cancela. [Fuente](https://numpy.org/doc/stable/reference/generated/numpy.var.html)

- **Varianza explicada** — Varianza explicada: Variación conservada por las componentes. En PCA, una componente conserva una cantidad de varianza. Su proporción explicada divide esa cantidad entre la varianza total de las variables preparadas; varias componentes conservan la suma de sus proporciones. 0,90 significa 90 % de esa varianza; no significa 90 % de aciertos ni garantiza conservar la información útil para predecir. Hay que declarar el centrado y escalado aplicados. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)


### Antes de mirar fórmulas: qué cambia al resumir

Primero dibujamos un punto por registro en tres ejes. Después los mismos 200 puntos se dibujan con dos números, PC1 y PC2, y finalmente con uno, PC1. **PC** significa componente principal. No son medidas recogidas de nuevo: se calculan con X1, X2 y X3.

**Estandarizar** pone cada variable en relación con su propio promedio y su dispersión, para que sus tamaños numéricos no dominen el resumen. La cuenta de una variable es (valor−media)/desviación. Con valor 60, media 50 y desviación 10 daría 1: una desviación por encima de la media. Esos tres números son un ejemplo de lectura, no estadísticas calculadas de esta simulación.

**Varianza** resume cuánto difieren los números respecto a su promedio. La **varianza conservada** informa cuánto de esa dispersión queda en el resumen. Si dos componentes conservaran 90 %, se descartaría 10 % de la variación de las variables preparadas. Eso no es 90 % de predicciones correctas. Lee el porcentaje que calcula tu ejecución, no uses el 100 % del ejemplo diagonal del aula.

**Para leer la tabla de pesos:** cada fila corresponde a una componente y cada columna a una variable original ya estandarizada. Un peso es el multiplicador que le da participación. Las direcciones son ortogonales: están a 90° entre sí en el espacio preparado. La multiplicación y suma se abrirá después con un solo registro, no necesitas interpretar toda la matriz a la vez.


In [ ]:
# ================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA (Colab listo para pegar)
# Caso simple: dataset sintético de 3 variables
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset con 3 variables correlacionadas.
# 2) Visualiza los datos en 3D.
# 3) Aplica PCA para reducir a 2D y a 1D.
# 4) Explica cómo leer los resultados: varianza explicada, componentes, proyecciones.
#
# ------------------------------------------------
# ¿Qué es PCA?
# ------------------------------------------------
# PCA = Principal Component Analysis.
# Objetivo: encontrar nuevas variables (componentes principales) que:
# - Son combinaciones lineales de las originales.
# - La primera maximiza varianza; las siguientes son ortogonales y capturan la restante.
# - Para un número fijo de dimensiones, minimizan el error de reconstrucción cuadrático.
# - La varianza conservada no garantiza conservar lo relevante para otra tarea.
#
# Ejemplo:
# - Si lluvia y humedad están muy correlacionadas, PCA crea una sola
#   componente que resume ambas, reduciendo redundancia.
# ================================================================

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D

# ----------------------------
# 1) Dataset sintético
# ----------------------------
# Creamos 3 variables con correlaciones:
# - X1 ~ normal
# - X2 ~ X1 con ruido (correlación alta)
# - X3 ~ independiente
rng = np.random.default_rng(42)
n = 200
X1 = rng.normal(50, 10, n)
X2 = X1 * 0.8 + rng.normal(0, 5, n) + 10
X3 = rng.normal(30, 8, n)

df = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})
display(df.head())

# ----------------------------
# 2) Visualización en 3D
# ----------------------------
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df["X1"], df["X2"], df["X3"], alpha=0.6)
ax.set_xlabel("X1")
ax.set_ylabel("X2")
ax.set_zlabel("X3")
ax.set_title("Datos originales en 3D")
plt.show()

# ----------------------------
# 3) Escalado
# ----------------------------
# PCA es sensible a escala → StandardScaler.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 4) PCA
# ----------------------------
pca = PCA(n_components=3)  # tantas como originales para analizar
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"Componente {i+1}: {v:.2%}")

print("\nVarianza acumulada:", np.cumsum(pca.explained_variance_ratio_))

print("\nComponentes (pesos aplicados a cada variable estandarizada en cada PC):")
print(pd.DataFrame(pca.components_, columns=df.columns, index=[f"PC{i+1}" for i in range(3)]))

# ----------------------------
# 5) Visualización PCA 2D
# ----------------------------
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Datos reducidos a 2D con PCA")
plt.grid(True)
plt.show()

# ----------------------------
# 6) Visualización PCA 1D
# ----------------------------
plt.figure(figsize=(6,2))
plt.scatter(X_pca[:,0], np.zeros_like(X_pca[:,0]), alpha=0.6)
plt.xlabel("PC1 (única dimensión)")
plt.yticks([])
plt.title("Datos reducidos a 1D con PCA")
plt.show()

# ----------------------------
# 7) Interpretación
# ----------------------------
print(f"""
================= CÓMO LEER LOS RESULTADOS =================
1) Varianza explicada:
   - PC1 captura la mayor proporción de varianza; consulta el porcentaje calculado.
   - PC2 añade otra parte significativa.
   - PC3 suele explicar poco si hay redundancia entre variables.
   → Decisión: consulta la varianza acumulada calculada para decidir cuántas componentes conservar.

2) Componentes (matriz de pesos):
   - Cada PC combina z1, z2, z3: zj = (Xj - mediaj) / desviaciónj.
   - PC1 = w1*z1 + w2*z2 + w3*z3; los pesos de la tabla actúan tras escalar.
   - Si PC1 tiene coeficientes altos y parecidos en X1 y X2,
     significa que resume la "información común" entre ambas.

3) Gráfica 2D:
   - PC1 y PC2 conservan {pca.explained_variance_ratio_[:2].sum():.2%} de la varianza estandarizada.
   - Se descarta {1-pca.explained_variance_ratio_[:2].sum():.2%}; no son las distancias originales sin escalar.

4) Gráfica 1D:
   - PC1 conserva {pca.explained_variance_ratio_[0]:.2%} y descarta {1-pca.explained_variance_ratio_[0]:.2%} de varianza.
   - Una sola dimensión puede perder estructura relevante: depende de la tarea.

5) Uso práctico:
   - PCA reduce variables redundantes.
   - Puede ayudar a visualizar y preparar datos para otros algoritmos.
   - Una mejora de clustering o regresión requiere comparación; no está garantizada.
=============================================================
""")


### Lee las figuras en este orden

**Primera figura, tres ejes:** cada punto tiene X1, X2 y X3. Es la lista original de 200 registros. Rotar mentalmente el dibujo no modifica los datos.

**Segunda figura, PC1 y PC2:** cada punto sigue siendo uno de esos registros, pero sus coordenadas son dos combinaciones de las variables preparadas. Busca en la salida «PC1 y PC2 conservan…»: ese porcentaje se calcula sumando las proporciones de ambas componentes. Su complemento hasta 100 % es lo descartado.

**Tercera figura, solo PC1:** los puntos se colocan sobre una línea. El eje vertical vale cero únicamente para dibujarlos; no significa que otra medida real sea cero. Dos puntos pueden caer cerca porque la información que los distinguía se descartó.

**Prueba pequeña sin cambiar datos:** compara en la salida el porcentaje de PC1 con el de PC1+PC2. El segundo no puede ser menor porque añade una componente. Una componente necesita un número por registro y dos necesitan dos: estás comparando tamaño del resumen frente a información conservada.


### Una ficha, una coordenada: seguir los números sin leer toda la matriz

`df.iloc[[0]]` toma únicamente el primer registro para poder seguirlo; no vuelve a ajustar PCA. La celda aplica tres pasos con los valores ya aprendidos:

1. `scaler.transform` resta a cada medida su media y divide por su desviación. La tabla muestra valor original, media y desviación para que puedas comprobarlo.
2. `pca.mean_` es el centro aprendido por PCA. Se resta al registro estandarizado; aquí es muy cercano a cero porque ya se estandarizó.
3. Cada número centrado se multiplica por el peso de PC1 situado en la misma columna. Los tres productos se suman: el resultado es la posición de este registro en el eje PC1.

**Cómo leer la salida:** recorre una variable de izquierda a derecha: Original → Media y Desviación → valor preparado → Peso PC1 → Producto. Después mira el texto bajo la tabla: «suma de productos» y «coordenada del gráfico» deben coincidir, salvo el redondeo de pantalla. `assert` es una comprobación automática: no imprime nada si coincide y detiene la celda si no.

**Tip:** un peso negativo no significa una medida mala; indica el sentido de su aporte al eje. El signo completo de una componente puede invertirse sin cambiar la varianza conservada. No redondees los números usados para la cuenta: redondea solo lo que muestras.


In [ ]:
fila_original = df.iloc[[0]]
fila_z = scaler.transform(fila_original)[0]
fila_centrada = fila_z - pca.mean_
productos = fila_centrada * pca.components_[0]
display(pd.DataFrame({'Variable':df.columns,'Original':fila_original.iloc[0].values,
                      'Media':scaler.mean_,'Desviación':scaler.scale_,
                      'z centrado':fila_centrada,'Peso PC1':pca.components_[0],
                      'Producto':productos}))
print('PC1 = suma de productos =',productos.sum(),'· coordenada del gráfico =',X_pca[0,0])
assert np.isclose(productos.sum(),X_pca[0,0])


### Lee, comprueba y explica

Sigue un punto desde sus tres medidas hasta PC1. Lee los porcentajes que calcula tu ejecución; no son los del ejemplo pequeño del aula.

**Prueba pequeña:** Una coordenada negativa indica posición respecto al centro. No significa rendimiento negativo ni mala calidad.

**Vuelve al caso:** Al reducir variables, informa la variación conservada y mantén la relación de cada punto con su registro original.

**Pregunta:** ¿Conservar 90 % de varianza significa lograr 90 % de exactitud?

**Respuesta razonada:** No. El porcentaje describe dispersión de las entradas conservada por la proyección; no compara predicciones con respuestas reales. Incluso una dirección con poca variación podría ser útil para una tarea predictiva.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-pca)


## Tema 3 · Comparar modelos sobre la misma prueba

**Lo que vas a poder hacer:** Seleccionar un modelo candidato mediante una comparación reproducible.

Combinar estimaciones puede compensar errores; midamos el resultado antes de confiar.

### Entiende la idea

Para planear transporte puedes pedir estimaciones a más de un modelo. Un modelo es una regla que usa información disponible para calcular una respuesta, por ejemplo cuántas toneladas se cosecharán. Un ensamble combina varias de esas respuestas. Empezaremos por sacar el promedio de dos estimaciones: sumarlas y dividir entre dos. Eso puede compensar errores distintos; no elimina un error que ambos cometan igual. Al conocer la cosecha compararemos cuánto se alejó cada estimación. Más modelos no garantizan una mejor respuesta.

### Un caso pequeño antes del programa

**Dos errores pueden compensarse**

Una cooperativa necesita estimar cuánto recogerá para preparar transporte. La tabla contiene tres fincas ficticias reservadas, su rendimiento observado y predicciones dadas de dos modelos imaginarios. Las cantidades están en toneladas por hectárea. Para hacer tangible la salida, supongamos además que cada finca tiene 10 ha: una estimación de 3 t/ha correspondería a 30 toneladas. Las predicciones no se reajustan usando estas respuestas.

| Finca | Observado (t/ha) | Modelo A (t/ha) | Modelo B (t/ha) | Promedio (t/ha) |
| --- | --- | --- | --- | --- |
| 1 | 3,0 | 2,0 | 4,0 | 3,0 |
| 2 | 4,0 | 5,0 | 3,0 | 4,0 |
| 3 | 5,0 | 5,0 | 5,0 | 5,0 |

1. En la finca 1, A estima 2 y B estima 4 t/ha. Su promedio es (2+4)/2 = 3 t/ha, igual a lo observado. Para 10 ha, A daría 20 t, B 40 t y la combinación 30 t.
2. En las tres fincas, los errores absolutos de A son 1, 1 y 0; los de B también. Cada MAE es (1+1+0)/3 = 0,67 t/ha aproximadamente. La combinación tiene errores 0, 0 y 0.
3. En este ejemplo construido los errores se compensan exactamente. Con dos predicciones de 2 t/ha para una finca observada de 3, el promedio seguiría en 2: combinar no eliminaría el error compartido.

**Qué demuestra:** En estas tres fincas, el promedio predice 3, 4 y 5 t/ha y su MAE es cero. A y B se compensan aquí; si compartieran el mismo error, el promedio también lo conservaría.

### Ahora identifica los datos de esta práctica

Compara las barras de MAE sobre las mismas 60 fincas reservadas. Menor error significa que la estimación se acercó más al rendimiento observado.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** No compares una cifra de entrenamiento con otra de prueba. Todas las candidatas deben medirse sobre las mismas fincas y con las mismas unidades.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.


In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.


In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.


In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


### 1.5. Aprender una regresión con las tres entradas

Este segundo pipeline hace tres pasos en orden:

1. **Imputar:** completa una lluvia ausente con la mediana aprendida de las lluvias de entrenamiento.
2. **Estandarizar:** expresa cada entrada respecto a su media y su dispersión, aprendidas con entrenamiento. Así se manejan magnitudes diferentes; esto no añade información ni garantiza una mejora.
3. **Regresión lineal:** aprende un valor inicial y un peso para cada entrada para estimar rendimiento.

Para datos de prueba, el pipeline aplica las mismas reglas y los pesos ya aprendidos; no los recalcula. La relación de temperatura que inventamos es curva y esta regresión usa las tres variables sin potencias: puede quedar error por esa simplificación y por la variación aleatoria.

**Lee la salida:** la mediana está en mm y es la que usará para cualquier lluvia ausente. No tiene por qué ser 600 mm: ese número pertenecía al ejemplo ilustrativo de cuatro fincas de la presentación.


In [ ]:
lineal = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LinearRegression()
)
lineal.fit(X_tr, y_tr)
print(f'Mediana de lluvia aprendida: {lineal[0].statistics_[0]:.3f} mm')


### 1.6. Comparar predicciones con respuestas reservadas

`predict(X_te)` estima rendimientos para las 60 fincas de prueba. Después comparamos cada predicción con su rendimiento conocido, `y_te`. Llamamos **error absoluto** a la distancia entre ambos valores, sin signo; el **MAE** es el promedio de esas distancias: `MAE = promedio(|y − ŷ|)`, donde `ŷ` es la predicción.

**Cómo leer la tabla y el gráfico:**

- Ambos modelos se evalúan sobre las mismas 60 fincas. Cada barra muestra su MAE, no el rendimiento promedio.
- El eje vertical usa t/ha. Un MAE de 0,46 significa un error absoluto promedio cercano a 0,46 toneladas por hectárea en esta prueba; no que todas las fincas tengan ese error.
- La barra más baja corresponde al menor error medio. Compara la regresión con la referencia para saber si las entradas ayudaron en este ejemplo.
- La última tabla muestra cinco fincas de prueba: respuesta conocida, predicción de regresión y error absoluto de cada una.

**Alcance:** con una ejecución completa desde la semilla 42, la referencia obtiene aproximadamente **0,705 t/ha** y la regresión **0,459 t/ha**. La regresión mejora esta referencia en esta simulación. No demuestra que funcionará igual en fincas reales, ni que una variable cause cambios en el rendimiento. Las cifras pueden cambiar si alteras los datos o ejecutas celdas aleatorias de nuevo sin reiniciar el recorrido.


In [ ]:
pred_base = base.predict(X_te)
pred_lineal = lineal.predict(X_te)
resultados = pd.DataFrame({
    'modelo': ['Media de entrenamiento', 'Regresión lineal'],
    'MAE_t_ha': [mean_absolute_error(y_te, pred_base),
                 mean_absolute_error(y_te, pred_lineal)]
})
display(resultados)
plt.bar(resultados.modelo, resultados.MAE_t_ha)
plt.ylabel('MAE en prueba (t/ha)')
plt.title('Comparar con una referencia: mismas 60 fincas de prueba')
plt.show()
comparacion = pd.DataFrame({
    'rendimiento_conocido_t_ha': y_te,
    'prediccion_lineal_t_ha': pred_lineal,
    'error_absoluto_t_ha': np.abs(y_te - pred_lineal)
})
display(comparacion.head().round(3))


### Lectura del gráfico 1 · ¿Las entradas mejoraron la referencia?

**Eje horizontal:** los dos modelos. **Eje vertical:** error absoluto medio en las 60 fincas de prueba, expresado en t/ha. La altura de una barra representa error, no producción ni número de fincas. Cada barra resume 60 comparaciones entre respuesta conocida y predicción.

En esta ejecución, la referencia tiene **0,705 t/ha** de MAE y la regresión **0,459 t/ha**. La segunda barra es más baja: la regresión comete menos error medio sobre la misma prueba. Eso es lo que podemos concluir para este ejemplo.

En la última tabla, una finca tiene rendimiento conocido **4,338**, predicción **4,889** y error absoluto **0,551 t/ha**. La predicción quedó por encima de la respuesta; otra fila puede quedar por debajo. El MAE promedia las distancias sin compensar unos errores con otros. No es un porcentaje de aciertos ni el error de todas las fincas.


**Conexión con la presentación:** la diapositiva 6 resume aprender y predecir; aquí ves las filas, los pasos y el error que produce ese recorrido. La diapositiva 8 usa cuatro fincas ilustrativas para explicar la mediana; este laboratorio calcula su propia mediana con sus 180 fincas de entrenamiento.

Documentación de consulta: [imputación](https://scikit-learn.org/stable/modules/impute.html), [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [MAE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error).


### Encuentros 3 y 4: nombres completos de las medidas y métodos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **MAE** — MAE: Mean Absolute Error (Error absoluto medio). Promedio de las diferencias sin signo entre cada predicción y su respuesta observada: se resta, se elimina el signo y se promedian las distancias. Menor es mejor sobre los mismos casos y unidades; 0 indica coincidencia en todos esos casos. Conserva la unidad de la respuesta y no significa que cada caso tenga ese mismo error. Si se observan 4 y 10 cajas y se predicen 3 y 7, los errores absolutos son 1 y 3 cajas: MAE = (1+3)/2 = 2 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **RMSE** — RMSE: Root Mean Squared Error (Raíz del error cuadrático medio). Raíz cuadrada del promedio de diferencias al cuadrado entre predicción y respuesta observada. Vuelve a las unidades de la respuesta, aunque conserva la mayor influencia de errores grandes del MSE. Menor es mejor sobre los mismos casos; no es el promedio de las distancias absolutas. Con errores de 1 y 3 cajas, MSE = 5 cajas² y RMSE = √5 ≈ 2,24 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)

- **R²** — R²: Coeficiente de determinación. Símbolo de una medida de regresión: R² = 1 − suma de errores cuadrados / suma de diferencias cuadradas respecto a la media observada. Para respuestas no constantes, 1 es ajuste perfecto y 0 rinde lo mismo que predecir su media; puede ser negativo. No es porcentaje de aciertos; con respuestas constantes el cociente no está definido y algunas bibliotecas sustituyen el resultado. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

- **DBSCAN** — DBSCAN: Density-Based Spatial Clustering of Applications with Noise (Agrupación espacial basada en densidad con casos no agrupados). Método que forma grupos conectando zonas con suficientes vecinos. eps fija la distancia máxima entre vecinos y min_samples el mínimo para que un punto sea núcleo, contando al propio punto. No se elige de antemano el número de grupos; −1 identifica puntos sin grupo con esa configuración. El radio no limita la distancia entre todos los miembros de un grupo y quedar aparte no demuestra mala calidad. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)


In [ ]:
modelos={'Referencia':base,'Árbol':make_pipeline(SimpleImputer(strategy='median'),DecisionTreeRegressor(max_depth=5,random_state=42)),'Random Forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42)),'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingRegressor(random_state=42))}
filas=[]
for nombre,modelo in modelos.items():
    modelo.fit(X_tr,y_tr);p=modelo.predict(X_te)
    filas.append({'modelo':nombre,'MAE_t_ha':mean_absolute_error(y_te,p),'RMSE_t_ha':np.sqrt(mean_squared_error(y_te,p)),'R2':r2_score(y_te,p)})
tabla=pd.DataFrame(filas);display(tabla)
plt.barh(tabla.modelo,tabla.MAE_t_ha);plt.xlabel('MAE en prueba (t/ha)');plt.title('Mismos datos de prueba para todos');plt.show()


### Lectura del gráfico 3 · ¿Qué estimador se acercó más?

Cada barra horizontal corresponde a un modelo; su longitud es **error**, no cosecha. El eje horizontal usa toneladas por hectárea (t/ha). Más corta significa menor error medio en las mismas 60 fincas de prueba. El orden vertical no clasifica la calidad.

**MAE, antes de su sigla:** para cada finca toma la diferencia entre lo observado y lo predicho sin signo; después promedia esas diferencias. Si los errores fueran 1 y 3 t/ha, su MAE sería (1+3)/2 = 2 t/ha. Son dos números de explicación, no el resultado de estas 60 fincas.

Con la ejecución completa de referencia se obtuvieron aproximadamente 0,705 para referencia, 0,437 para árbol, 0,358 para Random Forest y 0,301 t/ha para boosting. **Lee la tabla de tu ejecución**: si cambiaste datos, configuraciones o versiones, no copies esos números como si fueran salidas nuevas.

La tabla también muestra **RMSE**: eleva cada error al cuadrado, promedia y toma la raíz; conserva t/ha y hace pesar más los errores grandes. **R²** compara el error cuadrático con predecir la media de las respuestas de prueba; puede ser negativo y no es porcentaje de aciertos. El gráfico representa solo MAE: no intentes leer RMSE o R² en la longitud de sus barras.

**Tip para no cambiar dos cosas a la vez:** guarda la tabla actual. Cambia únicamente la profundidad máxima del árbol de 5 a 3 y vuelve a ejecutar este bloque, conservando X_tr/X_te. Cambian el árbol y sus errores; los casos de prueba son los mismos. No tiene que mejorar. La profundidad limita cuántas preguntas encadena y es una configuración elegida, no un valor aprendido.

Elegir repetidamente configuraciones por esta prueba acabaría usándola para desarrollar el modelo. En una aplicación se necesitaría validación durante el desarrollo y una prueba final separada. Este bloque muestra la comparación, no autoriza una selección de producción.


### Lee, comprueba y explica

Compara las barras de MAE sobre las mismas 60 fincas reservadas. Menor error significa que la estimación se acercó más al rendimiento observado.

**Prueba pequeña:** No compares una cifra de entrenamiento con otra de prueba. Todas las candidatas deben medirse sobre las mismas fincas y con las mismas unidades.

**Vuelve al caso:** Compara modelo y combinación con la misma respuesta, los mismos casos reservados y la misma métrica.

**Pregunta:** ¿Un bosque de muchos árboles garantiza superar a un árbol?

**Respuesta razonada:** No. La combinación puede reducir algunos errores, pero depende de los datos y de cuánto se parezcan las equivocaciones. Se decide mediante validación; la prueba final mide el resultado elegido.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-ensambles)


## Cierra el encuentro con una explicación

1. **Parecidos no son categorías de calidad:** Un grupo significa cercanía con las variables y parámetros elegidos. Antes de darle un significado operativo, revisa qué comparten sus registros.
2. **Resumir sin inventar clases:** Al reducir variables, informa la variación conservada y mantén la relación de cada punto con su registro original.
3. **Varias respuestas, una comparación justa:** Compara modelo y combinación con la misma respuesta, los mismos casos reservados y la misma métrica.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
